In [ ]:
import io
import json
import itertools
import numpy
import pandas
import matplotlib.pyplot as plt
import matplotlib.transforms
import seaborn
import jupyter_bbox_widget

from scipy.stats import spearmanr
from sklearn.metrics import r2_score

### prepare data

In [ ]:
subset = 1000

#### read token probabilities
the parquet is made by running the `scripts/convert_surprisal_dataset.py` script on Jaap's surprisal data file. Also available [ready-made](https://martijn.at/surprisal_test_0.3.parquet)

In [ ]:
lower_bound = pandas.read_parquet("../resources/surprisal/surprisal_test_0.3.parquet").rename(columns={'surprisal': 'log_likelyhood'}).explode('log_likelyhood').query(f"sentence < {subset}")

#### add part-of-speech data
this parquet is generated by the `scripts/build_pos_dataset.py` script and the test tree file. Also available [ready-made](https://martijn.at/test_tree_0.3.parquet)

In [ ]:
lower_bound['log_likelyhood'] = -lower_bound['log_likelyhood']
posdf = pandas.read_parquet("../resources/export/0.3/test_tree_0.3.parquet")
lower_bound = pandas.concat([lower_bound, posdf.query(f"sent < {subset}").reset_index(names="token_pos")[['token_pos', 'pos']]], axis=1)

#### read in the perplexities for every checkpoint
(generated with `src/lm_training/evaluate_ppl.py`)


(get my data for [transformer](https://martijn.at/gpt-ppl.tar.gz) and [xlstm](https://martijn.at/xlstm-ppl.tar.gz)

In [ ]:
together = []
for cp in range(200,3900,200):
    with open(f"../resources/checkpoints/hf/xlstm/saveoften/0.3/bos/b4e64/checkpoint-{cp}/ppl_eval.json", "r") as fh:
        ppl = json.load(fh)
        ppl10000 = list(itertools.chain.from_iterable(ppl['token_perplexities']))[:subset]
        df = pandas.DataFrame([{"probs": -numpy.log(i)} for i in ppl10000]).explode('probs').astype('float').reset_index(names="sent")
        df = (pandas
            .concat(
                    [df,lower_bound],
                    axis=1
                )
            .rename(columns=dict(
                    probs="model_results",
                    log_likelyhood="grammar_lowerbound")
                )
        )
        df['checkpoint'] = str(cp)
        together.append(df)
together = pandas.concat(together).reset_index(drop=True)


#### utility functions / definitions

In [ ]:
def draw_statistics(data, x, y, ax, lims, facet=False, **kwargs):
    if facet:
        title = ax.title.get_text()
        if "|" in title:
            col_title, row_title = title.split(" | ")
            col, col_val = col_title.split(" = ")
            row, row_val = row_title.split(" = ")
            data = data[(data[col] == col_val) & (data[row] == row_val)]
        else:
            col, col_val = title.split(" = ")
            data = data[(data[col] == col_val)]

    ax.text(lims[0], lims[1]-1, f"spearman corr.: {round(spearmanr(data[x], data[y]).correlation, 4)}")
    ax.text(lims[0], lims[1]-1.5, f"R²: {round(r2_score(data[x], data[y]), 4)}")

In [ ]:
def get_xflip_transform(figure):
# flip the X axis
    tt = matplotlib.transforms.Affine2D(numpy.array([[1,0,0],[0,-1,1],[0,0,1]]))
    xform = figure.transFigure.inverted() + tt + figure.transFigure
    return xform

In [ ]:
x0 = together['model_results'].min()
x1 = together['model_results'].max()
y0 = together['grammar_lowerbound'].min()
y1 = together['grammar_lowerbound'].max()

lims = [max(x0, y0), min(x1, y1)]
axis_vars = dict(data=together, x="model_results", y="grammar_lowerbound")


## Full data for each checkpoint

In [ ]:
global_axis_vars = axis_vars | dict(col="checkpoint", col_wrap=2)
g = seaborn.relplot(**global_axis_vars, alpha=0.05, height=8, aspect=1)
for ax in g.axes:
    ax.plot(lims, lims, "-r")
    draw_statistics(**global_axis_vars, ax=ax, lims=lims, facet=True)


## Select the checkpoint  you're interested in

In [ ]:
selcp = "1200"

## Select the area(s) to investigate

In [ ]:
cp_axis_vars = axis_vars | dict(data=together[together['checkpoint'] == selcp])
g = seaborn.relplot(**cp_axis_vars, alpha=0.05, height=8, aspect=1)
foo = g.ax.plot(lims, lims, "-r")
draw_statistics(**cp_axis_vars, ax=g.ax, lims=lims)
with io.BytesIO() as buffer:
    g.savefig(buffer)
    plt.close()
    w=jupyter_bbox_widget.BBoxWidget(image_bytes=buffer.getvalue(), classes="area")
display(w)

### convert the bounding box coordinates to dataset coordinates

In [ ]:
areas = [matplotlib.transforms.Bbox.from_bounds(*list(box.values())[:4]).extents for box in w.bboxes]
to_data = get_xflip_transform(g.figure) + g.ax.transData.inverted()

boxes = [
    matplotlib.transforms.Bbox.from_extents(
        numpy.sort(
            to_data.transform(area).reshape(2,-1),
            axis=0
    ).reshape(-1,4))

    for area in areas
]

subset = pandas.concat(
    [
        together[
            (together['model_results'] >= bb.x0) &
            (together['model_results'] <= bb.x1) &
            (together['grammar_lowerbound'] >= bb.y0) &
            (together['grammar_lowerbound'] <= bb.y1) &
            (together['checkpoint'] == selcp)
        ]
            for bb in boxes
    ])

with pandas.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(subset)
    display(subset.agg(dict(model_results=["min", "max"], grammar_lowerbound=["min", "max"])))

subset = subset[['sent', 'token_pos']]

### Sentences in the subset

In [ ]:
subset['sent'].unique()

In [ ]:
with pandas.option_context("display.max_rows", None, "display.max_colwidth", None): display(together[(together['sent'].isin(subset['sent'].unique())) & (together['checkpoint'] == "3800")].groupby("sent").agg(sentence=pandas.NamedAgg(column="token", aggfunc=lambda g: " ".join(g.to_list()))))

## plot the selected subset for each checkpoint

In [ ]:
testdf = subset.merge(together, how="left", on=subset.columns.tolist())
global_axis_vars = axis_vars | dict(col="checkpoint", col_wrap=3) | dict(data=testdf)
#g = seaborn.lmplot(**global_axis_vars, line_kws={'color': 'g'}, scatter_kws={'alpha': 0.05}, height=3, aspect=1)
g = seaborn.relplot(**global_axis_vars, alpha=0.05, height=3, aspect=1)
for ax in g.axes:
    ax.plot(lims, lims, "-r")
    #draw_statistics(**global_axis_vars, ax=ax, lims=lims, facet=True)